In [10]:
import librosa
import numpy as np
from scipy.signal import correlate

# 1. 파일 로드 (경로가 맞는지 확인해 주세요)
main_path = "03 Disney Fun to Read Set 1-01 - Just Like Me Reading 2.mp3"
template_path = "book_page_turn_34s.mp3"

print("1. 오디오 파일 로딩 중...")
y_main, sr = librosa.load(main_path, sr=22050, mono=True)
y_template, _ = librosa.load(template_path, sr=sr, mono=True)

print(f"   - 메인 오디오 길이: {len(y_main)/sr:.2f}초")
print(f"   - 템플릿 오디오 길이: {len(y_template)/sr:.2f}초")

# 2. 정규화
main_std = np.std(y_main)
temp_std = np.std(y_template)

main_norm = (y_main - np.mean(y_main)) / main_std if main_std > 0 else y_main
temp_norm = (
    (y_template - np.mean(y_template)) / temp_std if temp_std > 0 else y_template
)

# 3. Cross-Correlation 계산
print("2. 상관계수 연산 중...")
corr = correlate(main_norm, temp_norm, mode="valid") / len(y_template)

# 4. 수치 즉시 출력
print("\n==========================================")
print(f"★ 최고 상관계수 (Max): {np.max(corr):.6f}")
print(f"★ 평균 상관계수 (Mean): {np.mean(corr):.6f}")
print(f"★ 최소 상관계수 (Min): {np.min(corr):.6f}")
print("==========================================\n")

1. 오디오 파일 로딩 중...
   - 메인 오디오 길이: 195.81초
   - 템플릿 오디오 길이: 1.10초
2. 상관계수 연산 중...

★ 최고 상관계수 (Max): 0.299074
★ 평균 상관계수 (Mean): -0.000000
★ 최소 상관계수 (Min): -0.107448



In [12]:
import os
import librosa
import numpy as np
from pydub import AudioSegment
from scipy.signal import correlate, find_peaks


def auto_split_page_turns(
    main_audio_path,
    template_path,
    output_dir="./output_mp3",
    sensitivity=0.6,  # 감도 조절 (0.5: 깐깐함 ~ 0.8: 넉넉함)
    min_gap_sec=6.0,  # 책장 넘김 최소 간격
    remove_before_sec=0.1,
    remove_after_sec=0.2,
    bitrate="192k",
):
    os.makedirs(output_dir, exist_ok=True)

    # 1. 로드 및 신호 분석
    y_main, sr = librosa.load(main_audio_path, sr=22050, mono=True)
    y_template, _ = librosa.load(template_path, sr=sr, mono=True)
    template_len = len(y_template)

    # 2. 정규화 및 Cross-Correlation 연산
    main_std = np.std(y_main)
    temp_std = np.std(y_template)

    main_norm = (
        (y_main - np.mean(y_main)) / main_std if main_std > 0 else y_main
    )
    temp_norm = (
        (y_template - np.mean(y_template)) / temp_std if temp_std > 0 else y_template
    )

    corr = correlate(main_norm, temp_norm, mode="valid") / template_len

    # 3. 상관계수 통계치 우선 추출
    c_max = np.max(corr)
    c_min = np.min(corr)
    c_mean = np.mean(corr)
    c_std = np.std(corr)

    print("==========================================")
    print(f"★ 최고 상관계수 (Max)  : {c_max:.6f}")
    print(f"★ 평균 상관계수 (Mean) : {c_mean:.6f}")
    print(f"★ 최소 상관계수 (Min)  : {c_min:.6f}")
    print(f"★ 표준편차 (Std Dev)  : {c_std:.6f}")

    # 4. 동적 임계값(Dynamic Threshold) 유기적 산출
    # 평균과의 차이(Peak Noise) 대비 Max 값 위치를 고려하여 계산
    dynamic_threshold = c_mean + (c_max - c_mean) * sensitivity
    print(f"➜ 유기적으로 적용된 임계값: {dynamic_threshold:.6f}")
    print("==========================================")

    # 5. 동적 임계값 기반 피크 탐지
    min_distance_samples = int(min_gap_sec * sr)
    peaks, _ = find_peaks(
        corr, height=dynamic_threshold, distance=min_distance_samples
    )

    print(
        f"[탐지 완료] 총 {len(peaks)}개의 책장 넘김 구간이 감지되었습니다."
    )

    if len(peaks) == 0:
        print(
            "탐지된 구간이 없습니다. sensitivity 파라미터를 조금 낮춰보세요 (예: 0.5)."
        )
        return []

    # 6. MP3 분할 및 저장
    audio = AudioSegment.from_file(main_audio_path, format="mp3")
    last_ms = 0
    saved_files = []

    for i, cut_sample in enumerate(peaks):
        cut_time_sec = cut_sample / sr

        start_trim_ms = max(0, int((cut_time_sec - remove_before_sec) * 1000))
        end_trim_ms = min(
            len(audio),
            int(
                (cut_time_sec + remove_after_sec + (template_len / sr)) * 1000
            ),
        )

        if start_trim_ms > last_ms:
            chunk = audio[last_ms:start_trim_ms]
            out_path = os.path.join(output_dir, f"segment_{i+1:03d}.mp3")
            chunk.export(out_path, format="mp3", bitrate=bitrate)
            saved_files.append(out_path)

        last_ms = end_trim_ms

    if last_ms < len(audio):
        chunk = audio[last_ms:]
        out_path = os.path.join(output_dir, f"segment_{len(peaks)+1:03d}.mp3")
        chunk.export(out_path, format="mp3", bitrate=bitrate)
        saved_files.append(out_path)

    return saved_files


# 실행 예시
# auto_split_page_turns("main_audio.mp3", "page_turn_template.mp3")

## 특정 MP3 파일에 대해 자동으로 책장 넘김 구간을 감지하고 분할하는 함수

In [13]:
import glob
import os
import librosa
import numpy as np
from pydub import AudioSegment
from scipy.signal import correlate, find_peaks


def auto_split_page_turns(
    main_audio_path,
    template_path,
    output_dir,
    sensitivity=0.6,
    min_gap_sec=6.0,
    remove_before_sec=0.1,
    remove_after_sec=0.2,
    bitrate="192k",
):
    """단일 MP3 파일을 분석하여 책장 소리 기반으로 동적 분할 후 저장"""
    os.makedirs(output_dir, exist_ok=True)

    # 1. 로드 및 신호 분석
    y_main, sr = librosa.load(main_audio_path, sr=22050, mono=True)
    y_template, _ = librosa.load(template_path, sr=sr, mono=True)
    template_len = len(y_template)

    # 2. 정규화 및 Cross-Correlation 연산
    main_std = np.std(y_main)
    temp_std = np.std(y_template)

    main_norm = (
        (y_main - np.mean(y_main)) / main_std if main_std > 0 else y_main
    )
    temp_norm = (
        (y_template - np.mean(y_template)) / temp_std if temp_std > 0 else y_template
    )

    corr = correlate(main_norm, temp_norm, mode="valid") / template_len

    # 3. 상관계수 통계치 추출 및 유기적 임계값 산출
    c_max = np.max(corr)
    c_min = np.min(corr)
    c_mean = np.mean(corr)
    c_std = np.std(corr)

    dynamic_threshold = c_mean + (c_max - c_mean) * sensitivity

    print(
        f"  ★ Max: {c_max:.4f} | Mean: {c_mean:.4f} | Min: {c_min:.4f} | Std: {c_std:.4f}"
    )
    print(f"  ➜ 산출된 유기적 임계값: {dynamic_threshold:.4f}")

    # 4. 피크 탐지
    min_distance_samples = int(min_gap_sec * sr)
    peaks, _ = find_peaks(
        corr, height=dynamic_threshold, distance=min_distance_samples
    )

    print(f"  [탐지 완료] 총 {len(peaks)}개의 책장 넘김 구간 감지")

    if len(peaks) == 0:
        print(
            "  ⚠️ 감지된 책장 넘김 구간이 없어 원본 파일을 분할하지 않았습니다."
        )
        return []

    # 5. MP3 분할 및 저장
    audio = AudioSegment.from_file(main_audio_path, format="mp3")
    last_ms = 0
    saved_files = []

    for i, cut_sample in enumerate(peaks):
        cut_time_sec = cut_sample / sr

        start_trim_ms = max(0, int((cut_time_sec - remove_before_sec) * 1000))
        end_trim_ms = min(
            len(audio),
            int(
                (cut_time_sec + remove_after_sec + (template_len / sr)) * 1000
            ),
        )

        if start_trim_ms > last_ms:
            chunk = audio[last_ms:start_trim_ms]
            out_path = os.path.join(output_dir, f"segment_{i+1:03d}.mp3")
            chunk.export(out_path, format="mp3", bitrate=bitrate)
            saved_files.append(out_path)

        last_ms = end_trim_ms

    if last_ms < len(audio):
        chunk = audio[last_ms:]
        out_path = os.path.join(output_dir, f"segment_{len(peaks)+1:03d}.mp3")
        chunk.export(out_path, format="mp3", bitrate=bitrate)
        saved_files.append(out_path)

    print(f"  [저장 완료] -> {output_dir}\n")
    return saved_files


def batch_process_mp3_folder(
    input_folder,
    template_path,
    output_root_dir="./split_results",
    sensitivity=0.6,
    min_gap_sec=6.0,
):
    """특정 폴더의 모든 MP3 파일을 순회하며 각각 별도 폴더를 만들어 분할 저장"""
    mp3_files = glob.glob(os.path.join(input_folder, "*.mp3"))

    if not mp3_files:
        print(f"❌ '{input_folder}' 폴더에 MP3 파일이 존재하지 않습니다.")
        return

    print(f"📂 총 {len(mp3_files)}개의 MP3 파일에 대한 분할 작업을 시작합니다.\n")

    for idx, mp3_file in enumerate(mp3_files, 1):
        # 파일명(확장자 제외) 추출하여 각각의 하위 폴더 생성
        file_name = os.path.splitext(os.path.basename(mp3_file))[0]
        file_output_dir = os.path.join(output_root_dir, file_name)

        print(
            f"=================================================="
        )
        print(
            f"[{idx}/{len(mp3_files)}] 처리 중: {os.path.basename(mp3_file)}"
        )
        print(
            f"=================================================="
        )

        try:
            auto_split_page_turns(
                main_audio_path=mp3_file,
                template_path=template_path,
                output_dir=file_output_dir,
                sensitivity=sensitivity,
                min_gap_sec=min_gap_sec,
            )
        except Exception as e:
            print(f"❌ 오류 발생 ({os.path.basename(mp3_file)}): {e}\n")

    print("🎉 모든 MP3 파일의 일괄 분할 처리가 완료되었습니다!")


# ==========================================
# 실행 부분 (원하는 경로로 수정 후 사용하세요)
# ==========================================
if __name__ == "__main__":
    batch_process_mp3_folder(
        input_folder="./input_mp3_folder",  # 대상 MP3 파일들이 있는 폴더 경로
        template_path="page_turn_template.mp3",  # 기준 책장 소리 템플릿 파일
        output_root_dir="./split_results",  # 저장될 최상위 결과 폴더 경로
        sensitivity=0.6,  # 0.5 ~ 0.8 사이 조절 (유기적 임계값 감도)
        min_gap_sec=6.0,  # 최소 간격 (초)
    )

📂 총 23개의 MP3 파일에 대한 분할 작업을 시작합니다.

[1/23] 처리 중: 03 Disney Fun to Read Set 1-01 - Just Like Me Reading 2.mp3
  ★ Max: 0.2991 | Mean: -0.0000 | Min: -0.1074 | Std: 0.0081
  ➜ 산출된 유기적 임계값: 0.1794
  [탐지 완료] 총 15개의 책장 넘김 구간 감지
  [저장 완료] -> ./split_results\03 Disney Fun to Read Set 1-01 - Just Like Me Reading 2

[2/23] 처리 중: 03 Disney Fun to Read Set 1-02 - Bug Stew Reading 2.mp3
  ★ Max: 0.3328 | Mean: 0.0000 | Min: -0.1179 | Std: 0.0078
  ➜ 산출된 유기적 임계값: 0.1997
  [탐지 완료] 총 15개의 책장 넘김 구간 감지
  [저장 완료] -> ./split_results\03 Disney Fun to Read Set 1-02 - Bug Stew Reading 2

[3/23] 처리 중: 03 Disney Fun to Read Set 1-03 - Toy to Toy Reading 2.mp3
  ★ Max: 0.3508 | Mean: 0.0000 | Min: -0.1250 | Std: 0.0071
  ➜ 산출된 유기적 임계값: 0.2105
  [탐지 완료] 총 15개의 책장 넘김 구간 감지
  [저장 완료] -> ./split_results\03 Disney Fun to Read Set 1-03 - Toy to Toy Reading 2

[4/23] 처리 중: 03 Disney Fun to Read Set 1-04 - As You Wish Reading 2.mp3
  ★ Max: 0.3765 | Mean: -0.0000 | Min: -0.1332 | Std: 0.0072
  ➜ 산출된 유기적 임계값: 0.2259
  [

## mp3파일을 앞 또는 뒤를 잘라서 저장하는 함수

In [ ]:
import glob
import os
from pydub import AudioSegment


def cut_mp3_folder_ms(
    input_folder,
    output_folder="./cut_output",
    trim_front_ms=0,  # 앞부분 자를 시간 (밀리초 단위, 예: 1500 = 1.5초)
    trim_back_ms=0,  # 뒷부분 자를 시간 (밀리초 단위, 예: 2500 = 2.5초)
    bitrate="192k",
):
    """지정한 폴더 내의 모든 MP3 파일에서 앞/뒤 특정 시간을 ms 단위로 잘라내어 '파일명_cut.mp3'로 저장합니다."""
    os.makedirs(output_folder, exist_ok=True)
    mp3_files = glob.glob(os.path.join(input_folder, "*.mp3"))

    if not mp3_files:
        print(f"❌ '{input_folder}' 폴더에 MP3 파일이 존재하지 않습니다.")
        return

    print(f"📂 총 {len(mp3_files)}개의 MP3 파일 자르기 작업을 시작합니다.")
    print(f"  - 앞부분 제거: {trim_front_ms} ms")
    print(f"  - 뒷부분 제거: {trim_back_ms} ms\n")

    for idx, mp3_file in enumerate(mp3_files, 1):
        filename = os.path.basename(mp3_file)
        name_without_ext = os.path.splitext(filename)[0]

        try:
            audio = AudioSegment.from_file(mp3_file, format="mp3")
            total_duration_ms = len(audio)

            # 시작 및 종료 지점 계산 (ms 단위 직관 적용)
            start_ms = min(trim_front_ms, total_duration_ms)
            end_ms = max(0, total_duration_ms - trim_back_ms)

            # 자르려는 시간이 원본 길이보다 긴 경우 처리
            if start_ms >= end_ms:
                print(
                    f"⚠️ [{idx}/{len(mp3_files)}] '{filename}': 자르려는 총 시간이 원본 길이({total_duration_ms}ms)보다 길어 건너뜁니다."
                )
                continue

            # ms 구간 잘라내기
            cut_audio = audio[start_ms:end_ms]

            # 파일명_cut.mp3 형태로 저장
            out_filename = f"{name_without_ext}_cut.mp3"
            out_path = os.path.join(output_folder, out_filename)

            cut_audio.export(out_path, format="mp3", bitrate=bitrate)

            print(
                f"✅ [{idx}/{len(mp3_files)}] {filename} -> {out_filename} (원본: {total_duration_ms}ms -> 변경: {len(cut_audio)}ms)"
            )

        except Exception as e:
            print(f"❌ 오류 발생 ({filename}): {e}")

    print(f"\n🎉 모든 파일 처리 완료! 저장 폴더: '{output_folder}'")


# ==========================================
# 실행 방법 예시 (ms 단위 적용)
# ==========================================
if __name__ == "__main__":
    # 1) 앞 500ms(0.5초), 뒤 1200ms(1.2초) 자르기
    cut_mp3_folder_ms(
        input_folder="./input_mp3_folder",
        output_folder="./cut_results",
        trim_front_ms=500,
        trim_back_ms=1200,
    )

## mp3파일을 앞 또는 뒤를 잘라서 저장하는 함수 _v2 

### 특정 폴더 밑의 모든 파일을 검색해서, 앞부분을 자르는 함수

In [15]:
import os
from pydub import AudioSegment


def cut_matching_mp3_in_tree(
    root_folder,
    target_filename,  # 검색할 파일 이름 (예: "segment_001.mp3" 또는 확장자 제외 "segment_001")
    trim_front_ms=0,
    trim_back_ms=0,
    bitrate="192k",
):
    """특정 루트 폴더 하위의 모든 디렉터리를 탐색하여 지정한 target_filename과 일치하는 MP3를 자른 뒤,

    해당 파일이 위치한 원본 폴더에 '파일명_cut_result.mp3'로 저장합니다.
    """
    # target_filename 확장자 처리 (입력값 편의성 보장)
    if not target_filename.endswith(".mp3"):
        target_filename += ".mp3"

    print(f"🔍 '{root_folder}' 이하 디렉터리 탐색 시작...")
    print(f"🎯 대상 파일명: {target_filename}")
    print(
        f"✂️  자르기 설정: 앞 {trim_front_ms}ms / 뒤 {trim_back_ms}ms\n"
    )

    matched_count = 0

    # os.walk를 활용한 트리 형태 재귀 탐색
    for current_dir, _, files in os.walk(root_folder):
        for filename in files:
            if filename == target_filename:
                matched_count += 1
                file_path = os.path.join(current_dir, filename)
                name_without_ext = os.path.splitext(filename)[0]

                try:
                    audio = AudioSegment.from_file(file_path, format="mp3")
                    total_duration_ms = len(audio)

                    start_ms = min(trim_front_ms, total_duration_ms)
                    end_ms = max(0, total_duration_ms - trim_back_ms)

                    if start_ms >= end_ms:
                        print(
                            f"⚠️ [{matched_count}] '{file_path}': 자르려는 시간이 원본({total_duration_ms}ms)보다 길어 스킵합니다."
                        )
                        continue

                    cut_audio = audio[start_ms:end_ms]

                    # 동일한 폴더 위치에 '파일명_cut_result.mp3' 생성
                    out_filename = f"{name_without_ext}_cut_result.mp3"
                    out_path = os.path.join(current_dir, out_filename)

                    cut_audio.export(out_path, format="mp3", bitrate=bitrate)

                    print(
                        f"✅ [{matched_count}] 저장 완료: {out_path} ({total_duration_ms}ms -> {len(cut_audio)}ms)"
                    )

                except Exception as e:
                    print(f"❌ 오류 발생 ({file_path}): {e}")

    if matched_count == 0:
        print(
            f"❌ '{target_filename}'과 일치하는 파일을 찾지 못했습니다."
        )
    else:
        print(f"\n🎉 총 {matched_count}개의 일치하는 파일 처리가 완료되었습니다!")


# ==========================================
# 실행 방법 예시
# ==========================================
if __name__ == "__main__":
    # 트리 폴더 전체를 뒤져서 "segment_001.mp3"를 찾고,
    # 앞 500ms, 뒤 1000ms를 자른 후 해당 위치에 "segment_001_cut_result.mp3"로 저장
    cut_matching_mp3_in_tree(
        root_folder="./split_results",  # 탐색할 최상위 트리 폴더
        target_filename="segment_002.mp3",  # 찾아서 변경할 파일명
        trim_front_ms=1200,  # 앞 자를 시간 (ms)
        trim_back_ms=0,  # 뒤 자를 시간 (ms)
    )

🔍 './split_results' 이하 디렉터리 탐색 시작...
🎯 대상 파일명: segment_002.mp3
✂️  자르기 설정: 앞 1200ms / 뒤 0ms

✅ [1] 저장 완료: ./split_results\03 Disney Fun to Read Set 1-01 - Just Like Me Reading 2\segment_002_cut_result.mp3 (12429ms -> 11229ms)
✅ [2] 저장 완료: ./split_results\03 Disney Fun to Read Set 1-02 - Bug Stew Reading 2\segment_002_cut_result.mp3 (13784ms -> 12584ms)
✅ [3] 저장 완료: ./split_results\03 Disney Fun to Read Set 1-03 - Toy to Toy Reading 2\segment_002_cut_result.mp3 (6454ms -> 5254ms)
✅ [4] 저장 완료: ./split_results\03 Disney Fun to Read Set 1-04 - As You Wish Reading 2\segment_002_cut_result.mp3 (9884ms -> 8684ms)
✅ [5] 저장 완료: ./split_results\03 Disney Fun to Read Set 1-05 - Piglet Feels Small Reading 2\segment_002_cut_result.mp3 (11197ms -> 9997ms)
✅ [6] 저장 완료: ./split_results\03 Disney Fun to Read Set 1-06 - Big Friend, Little Friend Reading 2\segment_002_cut_result.mp3 (10265ms -> 9065ms)
✅ [7] 저장 완료: ./split_results\03 Disney Fun to Read Set 1-07 - Kingdom of Color Reading 2\segment_002_cu

In [17]:
if __name__ == "__main__":
    # 트리 폴더 전체를 뒤져서 "segment_001.mp3"를 찾고,
    # 앞 500ms, 뒤 1000ms를 자른 후 해당 위치에 "segment_001_cut_result.mp3"로 저장
    cut_matching_mp3_in_tree(
        root_folder="./split_results_12",  # 탐색할 최상위 트리 폴더
        target_filename="segment_012.mp3",  # 찾아서 변경할 파일명
        trim_front_ms=0,  # 앞 자를 시간 (ms)
        trim_back_ms=17000,  # 뒤 자를 시간 (ms)
    )

🔍 './split_results_12' 이하 디렉터리 탐색 시작...
🎯 대상 파일명: segment_012.mp3
✂️  자르기 설정: 앞 0ms / 뒤 17000ms

✅ [1] 저장 완료: ./split_results_12\03 Disney Fun to Read Set 1-23 - Fast Kart, Slow Kart Reading 2\segment_012_cut_result.mp3 (21829ms -> 4829ms)

🎉 총 1개의 일치하는 파일 처리가 완료되었습니다!


In [18]:
if __name__ == "__main__":
    # 트리 폴더 전체를 뒤져서 "segment_001.mp3"를 찾고,
    # 앞 500ms, 뒤 1000ms를 자른 후 해당 위치에 "segment_001_cut_result.mp3"로 저장
    cut_matching_mp3_in_tree(
        root_folder="./split_results_13",  # 탐색할 최상위 트리 폴더
        target_filename="segment_013.mp3",  # 찾아서 변경할 파일명
        trim_front_ms=0,  # 앞 자를 시간 (ms)
        trim_back_ms=17000,  # 뒤 자를 시간 (ms)
    )

🔍 './split_results_13' 이하 디렉터리 탐색 시작...
🎯 대상 파일명: segment_013.mp3
✂️  자르기 설정: 앞 0ms / 뒤 17000ms

✅ [1] 저장 완료: ./split_results_13\03 Disney Fun to Read Set 1-17 - Fame in the Fast Lane Reading 2\segment_013_cut_result.mp3 (24201ms -> 7201ms)
✅ [2] 저장 완료: ./split_results_13\03 Disney Fun to Read Set 1-18 - M Is for Monster Reading 2\segment_013_cut_result.mp3 (23163ms -> 6163ms)

🎉 총 2개의 일치하는 파일 처리가 완료되었습니다!


In [19]:
if __name__ == "__main__":
    # 트리 폴더 전체를 뒤져서 "segment_001.mp3"를 찾고,
    # 앞 500ms, 뒤 1000ms를 자른 후 해당 위치에 "segment_001_cut_result.mp3"로 저장
    cut_matching_mp3_in_tree(
        root_folder="./split_results_16",  # 탐색할 최상위 트리 폴더
        target_filename="segment_016.mp3",  # 찾아서 변경할 파일명
        trim_front_ms=0,  # 앞 자를 시간 (ms)
        trim_back_ms=17000,  # 뒤 자를 시간 (ms)
    )

🔍 './split_results_16' 이하 디렉터리 탐색 시작...
🎯 대상 파일명: segment_016.mp3
✂️  자르기 설정: 앞 0ms / 뒤 17000ms

✅ [1] 저장 완료: ./split_results_16\03 Disney Fun to Read Set 1-01 - Just Like Me Reading 2\segment_016_cut_result.mp3 (23016ms -> 6016ms)
✅ [2] 저장 완료: ./split_results_16\03 Disney Fun to Read Set 1-02 - Bug Stew Reading 2\segment_016_cut_result.mp3 (24180ms -> 7180ms)
✅ [3] 저장 완료: ./split_results_16\03 Disney Fun to Read Set 1-03 - Toy to Toy Reading 2\segment_016_cut_result.mp3 (23654ms -> 6654ms)
✅ [4] 저장 완료: ./split_results_16\03 Disney Fun to Read Set 1-04 - As You Wish Reading 2\segment_016_cut_result.mp3 (23272ms -> 6272ms)
✅ [5] 저장 완료: ./split_results_16\03 Disney Fun to Read Set 1-05 - Piglet Feels Small Reading 2\segment_016_cut_result.mp3 (24685ms -> 7685ms)
✅ [6] 저장 완료: ./split_results_16\03 Disney Fun to Read Set 1-06 - Big Friend, Little Friend Reading 2\segment_016_cut_result.mp3 (21643ms -> 4643ms)
✅ [7] 저장 완료: ./split_results_16\03 Disney Fun to Read Set 1-07 - Kingdom of Color

In [22]:
if __name__ == "__main__":
    # 트리 폴더 전체를 뒤져서 "segment_001.mp3"를 찾고,
    # 앞 500ms, 뒤 1000ms를 자른 후 해당 위치에 "segment_001_cut_result.mp3"로 저장
    cut_matching_mp3_in_tree(
        root_folder="./split_results_18",  # 탐색할 최상위 트리 폴더
        target_filename="segment_018.mp3",  # 찾아서 변경할 파일명
        trim_front_ms=0,  # 앞 자를 시간 (ms)
        trim_back_ms=17000,  # 뒤 자를 시간 (ms)
    )

🔍 './split_results_18' 이하 디렉터리 탐색 시작...
🎯 대상 파일명: segment_018.mp3
✂️  자르기 설정: 앞 0ms / 뒤 17000ms

✅ [1] 저장 완료: ./split_results_18\03 Disney Fun to Read Set 1-10 - Alice in Wonderland Reading 2\segment_018_cut_result.mp3 (25094ms -> 8094ms)
✅ [2] 저장 완료: ./split_results_18\03 Disney Fun to Read Set 1-16 - Beauty and the Beast Reading 2\segment_018_cut_result.mp3 (24910ms -> 7910ms)
✅ [3] 저장 완료: ./split_results_18\03 Disney Fun to Read Set 1-19 - Up, Up, and Away! Reading 2\segment_018_cut_result.mp3 (24520ms -> 7520ms)

🎉 총 3개의 일치하는 파일 처리가 완료되었습니다!
